In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [ ]:
%%file gdansk_najnizsza_srednia.py
from pyspark.sql.functions import to_timestamp, col, window, avg, round as _round

df = spark.read.json("data/transactions_10k.jsonl")
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print("Godzina, w której Gdańsk miał najniższą średnią")

gdansk_srednia_na_h = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_PLN"))
    .select(
        col("window.start").alias("godzina_od"),
        col("window.end").alias("godzina_do"),
        "srednia_PLN"
    )
    .orderBy("srednia_PLN")
)
gdansk_hourly_avg.show(1, truncate=False)

In [ ]:
%%file liczba_trans_9_do_930.py

print("Zadanie 2: Liczba transakcji na kategorie w oknie 09:00 - 09:30")

kategoria_w_oknie = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("liczba_tx"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "category",
        "liczba_tx"
    )
    .filter(col("od") == "2026-02-14 09:00:00")
    .orderBy("category")
)
kategoria_w_oknie.show(truncate=False)



In [ ]:
%%file kwadranse.py
from pyspark.sql.functions import to_timestamp, col, window, count, desc

print("Zadanie 3: Szczyt transakcji w kwadransach")
trans_suma_kwadranse= (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .orderBy(desc("liczba_tx"))
)
trans_suma_kwadranse.show(1, truncate=False)

spark.stop()